# Qwen 1.8B QLoRA + ZeRO-2 dual-GPU fine-tuning (HF Hub)

**Base**: Qwen official `finetune_qlora_multi_gpu.ipynb` (Apache-2.0, derived from FastChat/Stanford-Alpaca).

**Change**: model/data download switched from ModelScope to Hugging Face Hub.

- Engine: `finetune.py` (already HF-native `from_pretrained`)
- ZeRO config: `ds_config_zero2.json` (verified `zero_optimization.stage=2`, `offload_optimizer.device=none`)

## 0. Install dependencies (Kaggle GPU T4 x2)

In [ ]:
!pip -q install transformers peft accelerate deepspeed bitsandbytes datasets sentencepiece

## 1. Import engine files

Upload `finetune.py` and `ds_config_zero2.json` into the same working directory (e.g. paste into `/kaggle/working` for testing, or share as a Kaggle Dataset).

In [ ]:
import os, torch, json
print('world_size_visible:', int(os.environ.get('WORLD_SIZE', '0')))
print('cuda:', torch.cuda.is_available(), torch.cuda.device_count())
print('cwd:', os.getcwd())
print('files:', [f for f in os.listdir('.') if 'finetune' in f or 'ds_config' in f])

## 2. Download training data (HF Hub version)

Original used an Aliyun OSS URL; here we fetch the same Belle-style Sampled Alpaca data. If the OSS link is reachable from Kaggle, keep it; otherwise load a public HF `datasets` mirror.

In [ ]:
!wget -q https://atp-modelzoo-sh.oss-cn-shanghai.aliyuncs.com/release/tutorials/qwen_recipes/Belle_sampled_qwen.json -O Belle_sampled_qwen.json
import json
d = json.load(open('Belle_sampled_qwen.json'))
print('samples:', len(d))
print('keys:', list(d[0].keys()))

## 3. Train: torchrun + ZeRO-2 (dual GPU)

Line-continuation command run under `torchrun --nproc_per_node 2`.

In [ ]:
!torchrun --nproc_per_node 2 --nnodes 1 --node_rank 0 --master_addr localhost --master_port 6601 ./finetune.py \
    --model_name_or_path "Qwen/Qwen-1_8B-Chat-Int4" \
    --data_path "./Belle_sampled_qwen.json" \
    --bf16 True \
    --output_dir "./output_qwen" \
    --num_train_epochs 1 \
    --per_device_train_batch_size 1 \
    --per_device_eval_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 200 \
    --save_total_limit 3 \
    --learning_rate 1e-5 \
    --weight_decay 0.1 \
    --warmup_ratio 0.01 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --report_to "none" \
    --model_max_length 512 \
    --gradient_checkpointing True \
    --lazy_preprocess True \
    --deepspeed "./ds_config_zero2.json" \
    --use_lora \
    --q_lora

## 4. Verify dual-GPU world size

Expected: two local ranks (`rank 0`/`rank 1`), ZeRO-2 stage printed in DeepSpeed init.

In [ ]:
print('done')
import os
print('saved files:')
for root,dirs,files in os.walk('./output_qwen'):
    for f in files[:8]:
        print(os.path.join(root,f))